# 🎯 Model 1 v2 — Customer Segmentation (Fixed)
### RobustScaler + Log Transform → Better clusters
**Problem fixed:** Extreme Monetary outliers (£580K) were skewing K-Means
**Solution:** RobustScaler + log1p transform before clustering

## Cell 1 — Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import RobustScaler
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import silhouette_score
import warnings
warnings.filterwarnings("ignore")

C = ["#534AB7","#1D9E75","#D85A30","#BA7517","#185FA5","#993C1D"]
SEG_COLORS = {"Champion":"#534AB7","Loyal":"#1D9E75","At-Risk":"#BA7517","Lost":"#D85A30"}
plt.rcParams.update({"figure.figsize":(14,5),"axes.spines.top":False,"axes.spines.right":False,"font.size":11})
print("Libraries ready")

## Cell 2 — Load RFM Data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

rfm = pd.read_csv("/content/drive/MyDrive/retail_project/data_rfm.csv")
print(f"Shape: {rfm.shape}")
print(f"\nRFM range (this shows WHY we need better scaling):")
print(f"  Recency  : {rfm['Recency'].min()} to {rfm['Recency'].max()}")
print(f"  Frequency: {rfm['Frequency'].min()} to {rfm['Frequency'].max()}")
print(f"  Monetary : £{rfm['Monetary'].min():.0f} to £{rfm['Monetary'].max():.0f}")
print(f"\nMonetary top 10 (these extreme values caused the problem):")
print(rfm[["Customer ID","Monetary"]].nlargest(10,"Monetary").to_string(index=False))

## Cell 3 — Why StandardScaler Failed + Our Fix

In [ ]:
# THE PROBLEM:
# Monetary max = £580,987 — this is 680x the median (£854)
# Even after StandardScaler, this extreme value still pulls K-Means to create:
#   - 4 "Champion" customers (the extreme outliers)
#   - Everyone else crammed into At-Risk and Lost

# OUR FIX: Two steps
# Step 1: log1p transform — compresses extreme values
#          log1p(854) = 6.75    log1p(580987) = 13.27  (much closer now!)
# Step 2: RobustScaler — uses median+IQR instead of mean+std
#          Less sensitive to remaining outliers

# Apply log1p first
rfm["R_log"] = np.log1p(rfm["Recency"])
rfm["F_log"] = np.log1p(rfm["Frequency"])
rfm["M_log"] = np.log1p(rfm["Monetary"])

print("After log1p transform:")
print(f"  Recency  : {rfm['R_log'].min():.2f} to {rfm['R_log'].max():.2f}")
print(f"  Frequency: {rfm['F_log'].min():.2f} to {rfm['F_log'].max():.2f}")
print(f"  Monetary : {rfm['M_log'].min():.2f} to {rfm['M_log'].max():.2f}")
print("\n→ Now Monetary range is 0-13 instead of £3-£580,987")
print("→ Much more balanced — K-Means will work fairly now")

## Cell 4 — Apply RobustScaler

In [ ]:
X_log = rfm[["R_log","F_log","M_log"]].copy()

# RobustScaler: uses median and IQR (not mean and std)
# Outliers have much less effect on RobustScaler
scaler = RobustScaler()
X_scaled = scaler.fit_transform(X_log)

print("After RobustScaler:")
print(f"  Median of each feature (should be ~0): {np.median(X_scaled, axis=0).round(3)}")
print(f"  IQR of each feature   (should be ~1): {(np.percentile(X_scaled,75,axis=0) - np.percentile(X_scaled,25,axis=0)).round(3)}")

# Compare distributions before vs after
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle("RFM After log1p + RobustScaler — Ready for K-Means", fontsize=12, fontweight="bold")
for i, (col, color, label) in enumerate(zip(
    ["R_log","F_log","M_log"], C[:3],
    ["Recency","Frequency","Monetary"]
)):
    axes[i].hist(X_scaled[:,i], bins=50, color=color, edgecolor="white", alpha=0.85)
    axes[i].axvline(0, color="black", lw=2, linestyle="--", label="Median=0")
    axes[i].set_title(f"{label} (scaled)")
    axes[i].legend(fontsize=9)
plt.tight_layout()
plt.show()

# OBSERVATION:
# Distributions now look much more balanced
# No single feature will dominate the clustering

## Cell 5 — Elbow Method (Best K)

In [ ]:
inertias   = []
sil_scores = []
K_range    = range(2, 11)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_scaled, km.labels_))
    print(f"  K={k}  |  Inertia={km.inertia_:,.0f}  |  Silhouette={sil_scores[-1]:.4f}")

best_k = list(K_range)[np.argmax(sil_scores)]
print(f"\nBest K by Silhouette: {best_k}")
print("We will use K=4 for business interpretability")

## Cell 6 — Elbow + Silhouette Chart

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Finding Best K — After Fix", fontsize=13, fontweight="bold")

axes[0].plot(list(K_range), inertias, "o-", color=C[0], lw=2.5, markersize=8)
axes[0].axvline(4, color=C[2], lw=2, linestyle="--", label="K=4 (our choice)")
axes[0].set_title("Elbow Method — Inertia")
axes[0].set_xlabel("K"); axes[0].set_ylabel("Inertia")
axes[0].legend()

axes[1].plot(list(K_range), sil_scores, "o-", color=C[1], lw=2.5, markersize=8)
axes[1].axvline(4, color=C[2], lw=2, linestyle="--", label="K=4 (our choice)")
axes[1].set_title("Silhouette Score (higher = better)")
axes[1].set_xlabel("K"); axes[1].set_ylabel("Silhouette Score")
axes[1].legend()

plt.tight_layout()
plt.show()

## Cell 7 — Fit K-Means K=4

In [ ]:
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10, max_iter=300)
rfm["Cluster"] = kmeans.fit_predict(X_scaled)

print(f"K-Means fitted — K=4")
print(f"Silhouette score: {silhouette_score(X_scaled, rfm['Cluster']):.4f}")
print(f"\nCustomers per cluster:")
print(rfm["Cluster"].value_counts().sort_index())

## Cell 8 — Analyze Clusters & Assign Labels

In [ ]:
# Look at the ORIGINAL (non-scaled) RFM means per cluster
cluster_means = rfm.groupby("Cluster")[["Recency","Frequency","Monetary"]].mean().round(2)
cluster_size  = rfm.groupby("Cluster").size().rename("Count")
cluster_summary = cluster_means.join(cluster_size)

print("Cluster profiles (original RFM values):")
print(cluster_summary)

# Score: Low Recency + High Frequency + High Monetary = best customer
cluster_summary["Score"] = (
    -cluster_summary["Recency"] +
    cluster_summary["Frequency"] * 10 +
    cluster_summary["Monetary"] / 100
)

# Assign labels by rank
cluster_summary["Rank"] = cluster_summary["Score"].rank(ascending=False).astype(int)
label_map = {}
for cid, rank in cluster_summary["Rank"].items():
    if rank == 1:   label_map[cid] = "Champion"
    elif rank == 2: label_map[cid] = "Loyal"
    elif rank == 3: label_map[cid] = "At-Risk"
    else:           label_map[cid] = "Lost"

rfm["Segment"] = rfm["Cluster"].map(label_map)

print("\nSegment assignment:")
for cid, seg in label_map.items():
    row = cluster_summary.loc[cid]
    print(f"  Cluster {cid} → {seg:<12} | Recency={row['Recency']:.0f}d | "
          f"Freq={row['Frequency']:.1f} | Monetary=£{row['Monetary']:.0f} | n={row['Count']}")

print("\nSegment distribution:")
seg_counts = rfm["Segment"].value_counts()
for seg, count in seg_counts.items():
    pct = count/len(rfm)*100
    bar = "█" * int(pct/2)
    print(f"  {seg:<12}: {count:>4} ({pct:.1f}%)  {bar}")

## Cell 9 — Visualize Segments

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle("Customer Segmentation — K-Means K=4 (Fixed Version)", fontsize=14, fontweight="bold")

# Chart 1: Distribution pie
seg_order  = ["Champion","Loyal","At-Risk","Lost"]
seg_counts_ord = rfm["Segment"].value_counts().reindex(seg_order)
axes[0,0].pie(
    seg_counts_ord.values,
    labels=seg_counts_ord.index,
    autopct="%1.1f%%",
    colors=[SEG_COLORS[s] for s in seg_order],
    startangle=90, explode=[0.05]*4
)
axes[0,0].set_title("Segment Distribution")

# Chart 2: Recency vs Monetary scatter
for seg, color in SEG_COLORS.items():
    mask = rfm["Segment"] == seg
    axes[0,1].scatter(rfm.loc[mask,"Recency"],
                      np.log1p(rfm.loc[mask,"Monetary"]),
                      c=color, label=seg, alpha=0.5, s=20)
axes[0,1].set_title("Recency vs log(Monetary) by Segment")
axes[0,1].set_xlabel("Recency (days) ← lower = more recent")
axes[0,1].set_ylabel("log(Monetary)")
axes[0,1].legend()

# Chart 3: Avg RFM profile per segment (normalized)
seg_means = rfm.groupby("Segment")[["Recency","Frequency","Monetary"]].mean()
seg_means = seg_means.reindex(seg_order)
seg_norm  = (seg_means - seg_means.min()) / (seg_means.max() - seg_means.min())
seg_norm.plot(kind="bar", ax=axes[1,0], color=C[:3], edgecolor="white", width=0.7)
axes[1,0].set_title("Normalized RFM Profile per Segment")
axes[1,0].set_xlabel(""); axes[1,0].set_ylabel("Normalized (0–1)")
axes[1,0].tick_params(axis="x", rotation=0)
axes[1,0].legend()

# Chart 4: Revenue contribution
seg_rev = rfm.groupby("Segment")["Monetary"].sum().reindex(seg_order)
bars = axes[1,1].bar(seg_rev.index,
                     seg_rev.values/1000,
                     color=[SEG_COLORS[s] for s in seg_order],
                     edgecolor="white")
axes[1,1].set_title("Total Revenue per Segment")
axes[1,1].set_ylabel("Total Revenue (£K)")
for i, (seg, val) in enumerate(seg_rev.items()):
    pct = val/seg_rev.sum()*100
    axes[1,1].text(i, val/1000+5, f"£{val/1000:.0f}K\n({pct:.1f}%)", ha="center", fontsize=9)

plt.tight_layout()
plt.show()

## Cell 10 — DBSCAN (Secondary Model)

In [ ]:
dbscan = DBSCAN(eps=0.5, min_samples=5)
rfm["DBSCAN_Cluster"] = dbscan.fit_predict(X_scaled)

n_noise = (rfm["DBSCAN_Cluster"] == -1).sum()
n_cls   = rfm["DBSCAN_Cluster"].nunique() - 1

print(f"DBSCAN: {n_cls} cluster(s) found")
print(f"Noise (anomalous customers): {n_noise} ({n_noise/len(rfm)*100:.1f}%)")
print(f"\nSample anomalous customers:")
noise = rfm[rfm["DBSCAN_Cluster"]==-1][["Customer ID","Recency","Frequency","Monetary","Segment"]].head(8)
print(noise.to_string(index=False))

# OBSERVATION:
# Noise customers = unusual behavior
# High Monetary + unusual Frequency = potential VIP or anomaly
# These feed into Model 5 (Anomaly Detection)

## Cell 11 — Segment Profiles + Business Actions

In [ ]:
print("=" * 65)
print("  SEGMENT PROFILES & BUSINESS ACTIONS")
print("=" * 65)

seg_profile = rfm.groupby("Segment").agg(
    Customers     = ("Customer ID",  "count"),
    Avg_Recency   = ("Recency",      "mean"),
    Avg_Frequency = ("Frequency",    "mean"),
    Avg_Monetary  = ("Monetary",     "mean"),
    Total_Revenue = ("Monetary",     "sum"),
    Churn_Rate    = ("IsChurned",    "mean"),
).round(2).reindex(["Champion","Loyal","At-Risk","Lost"])

actions = {
    "Champion": "VIP program. Ask for reviews. Early product access.",
    "Loyal"   : "Loyalty points. Referral program. Upsell.",
    "At-Risk" : "Win-back email with discount NOW. Create urgency.",
    "Lost"    : "Last-chance re-engagement. Big discount or survey.",
}

for seg in seg_profile.index:
    r = seg_profile.loc[seg]
    print(f"\n  {seg.upper()}")
    print(f"    Customers  : {int(r['Customers']):,} ({r['Customers']/len(rfm)*100:.1f}%)")
    print(f"    Avg Recency: {r['Avg_Recency']:.0f} days")
    print(f"    Avg Orders : {r['Avg_Frequency']:.1f}")
    print(f"    Avg Spend  : £{r['Avg_Monetary']:.0f}")
    print(f"    Total Rev  : £{r['Total_Revenue']:,.0f}")
    print(f"    Churned    : {r['Churn_Rate']*100:.0f}%")
    print(f"    Action     : {actions[seg]}")

print(f"\n{'='*65}")

## Cell 12 — Save Output (Agent 1 Output)

In [ ]:
output_cols = [c for c in [
    "Customer ID","Recency","Frequency","Monetary",
    "UniqueProducts","AvgOrderValue","TotalItems",
    "FavCategory","Country","CustomerLifetimeDays",
    "AvgDaysBetweenOrders","IsChurned","ReturnRate",
    "Cluster","Segment","DBSCAN_Cluster"
] if c in rfm.columns]

segments_out = rfm[output_cols].copy()

SAVE = "/content/drive/MyDrive/retail_project/"
segments_out.to_csv(SAVE + "data_segments.csv", index=False)

print(f"Saved: data_segments.csv")
print(f"  Rows    : {len(segments_out):,} customers")
print(f"  Columns : {segments_out.shape[1]}")
print(f"\nFinal segment breakdown:")
for seg, cnt in rfm["Segment"].value_counts().reindex(["Champion","Loyal","At-Risk","Lost"]).items():
    pct = cnt/len(rfm)*100
    avg_m = rfm[rfm["Segment"]==seg]["Monetary"].mean()
    print(f"  {seg:<12}: {cnt:>4} customers ({pct:.1f}%)  avg=£{avg_m:.0f}")

print(f"\nFeeds into:")
print(f"  Power BI  → Segment page")
print(f"  Agent 1   → LangGraph output")
print(f"  Model 3   → Churn (Segment as feature)")
print(f"  Model 6   → Inventory (FavCategory per segment)")

## Cell 13 — Model 1 Complete Summary

In [ ]:
sil = silhouette_score(X_scaled, rfm["Cluster"])

print(f"""
+----------------------------------------------------------+
|     MODEL 1 — CUSTOMER SEGMENTATION COMPLETE (v2)       |
+----------------------------------------------------------+
|  Algorithm : K-Means (K=4) + DBSCAN                     |
|  Scaler    : log1p + RobustScaler (fixed from v1)        |
|  Silhouette: {sil:.4f}  (0.4-0.6 = good)              |
|  Output    : data_segments.csv — 5,861 customers         |
+----------------------------------------------------------+""")

for seg, cnt in rfm["Segment"].value_counts().reindex(["Champion","Loyal","At-Risk","Lost"]).items():
    pct = cnt/len(rfm)*100
    avg_m = rfm[rfm["Segment"]==seg]["Monetary"].mean()
    print(f"|  {seg:<12}: {cnt:>4} ({pct:4.1f}%)  avg spend=£{avg_m:<8.0f}  |")

print("""|                                                          |
|  NEXT → Model 2: Demand Forecasting                      |
|  File  : data_ts_weekly.csv                              |
|  Models: ARIMA + SARIMA + Prophet                        |
+----------------------------------------------------------+
""")